In [0]:
-- ============================================================
-- GOLD VIEW: dashboard_metrics
-- PURPOSE:
--   Unified daily dashboard table for Looker Studio global filters.
--
--
-- BASE:
--   Sessions define the grain (prevents double counting).
--
-- INPUT TABLES (Silver):
--   - silver.sessions
--   - silver.page_views
--   - silver.conversion
--   - silver.campaigns
--
-- OUTPUT:
--   gold.dashboard_metrics
-- ============================================================

USE CATALOG shoply_analytics;
USE SCHEMA gold;

CREATE OR REPLACE VIEW dashboard_metrics AS
WITH

-- ============================================================
-- 0) BASE SESSIONS (GRAIN BUILDER)
-- ============================================================
base_sessions AS (
    SELECT
        DATE(session_start) AS date,

        -- campaign
        campaign_id,

        -- channel (normalized)
        COALESCE(NULLIF(LOWER(TRIM(utm_source)), ''), 'unknown') AS utm_source,
        COALESCE(NULLIF(LOWER(TRIM(utm_medium)), ''), 'unknown') AS utm_medium,
        COALESCE(NULLIF(LOWER(TRIM(utm_campaign)), ''), 'unknown') AS utm_campaign,

        -- device (normalized)
        COALESCE(NULLIF(LOWER(TRIM(device_type)), ''), 'unknown') AS device_type,

        COUNT(*) AS sessions,
        COUNT(DISTINCT user_id) AS unique_users
    FROM shoply_analytics.silver.sessions
    GROUP BY
        DATE(session_start),
        campaign_id,
        COALESCE(NULLIF(LOWER(TRIM(utm_source)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(utm_medium)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(utm_campaign)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(device_type)), ''), 'unknown')
),

-- ============================================================
-- 1) PAGE VIEWS (JOINED TO SESSIONS TO INHERIT DIMENSIONS)
-- ============================================================
pageviews AS (
    SELECT
        DATE(s.session_start) AS date,
        s.campaign_id,

        COALESCE(NULLIF(LOWER(TRIM(s.utm_source)), ''), 'unknown') AS utm_source,
        COALESCE(NULLIF(LOWER(TRIM(s.utm_medium)), ''), 'unknown') AS utm_medium,
        COALESCE(NULLIF(LOWER(TRIM(s.utm_campaign)), ''), 'unknown') AS utm_campaign,

        COALESCE(NULLIF(LOWER(TRIM(s.device_type)), ''), 'unknown') AS device_type,

        COUNT(*) AS page_views
    FROM shoply_analytics.silver.page_views p
    JOIN shoply_analytics.silver.sessions s
        ON p.session_id = s.session_id
    GROUP BY
        DATE(s.session_start),
        s.campaign_id,
        COALESCE(NULLIF(LOWER(TRIM(s.utm_source)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(s.utm_medium)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(s.utm_campaign)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(s.device_type)), ''), 'unknown')
),

-- ============================================================
-- 2) CONVERSIONS (JOINED TO SESSIONS TO INHERIT DIMENSIONS)
-- ============================================================
conversions AS (
    SELECT
        DATE(s.session_start) AS date,
        s.campaign_id,

        COALESCE(NULLIF(LOWER(TRIM(s.utm_source)), ''), 'unknown') AS utm_source,
        COALESCE(NULLIF(LOWER(TRIM(s.utm_medium)), ''), 'unknown') AS utm_medium,
        COALESCE(NULLIF(LOWER(TRIM(s.utm_campaign)), ''), 'unknown') AS utm_campaign,

        COALESCE(NULLIF(LOWER(TRIM(s.device_type)), ''), 'unknown') AS device_type,

        COUNT(*) AS conversions
    FROM shoply_analytics.silver.conversion c
    JOIN shoply_analytics.silver.sessions s
        ON c.session_id = s.session_id
    GROUP BY
        DATE(s.session_start),
        s.campaign_id,
        COALESCE(NULLIF(LOWER(TRIM(s.utm_source)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(s.utm_medium)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(s.utm_campaign)), ''), 'unknown'),
        COALESCE(NULLIF(LOWER(TRIM(s.device_type)), ''), 'unknown')
),

-- ============================================================
-- 3) CAMPAIGN METADATA
-- ============================================================
campaign_info AS (
    SELECT
        campaign_id,
        campaign_name,
        start_date,
        end_date,
        budget
    FROM shoply_analytics.silver.campaigns
)

-- ============================================================
-- 4) FINAL SELECT
-- ============================================================
SELECT
    bs.date,

    bs.campaign_id,
    ci.campaign_name,
    ci.start_date AS campaign_start_date,
    ci.end_date   AS campaign_end_date,
    ci.budget     AS campaign_budget,

    bs.utm_source,
    bs.utm_medium,
    bs.utm_campaign,

    bs.device_type,

    bs.sessions,
    bs.unique_users,
    pv.page_views,
    cv.conversions,

    CASE WHEN COALESCE(bs.sessions,0) > 0
         THEN COALESCE(pv.page_views,0) / bs.sessions
         ELSE NULL
    END AS avg_pages_per_session,

    CASE WHEN COALESCE(bs.sessions,0) > 0
         THEN COALESCE(cv.conversions,0) / bs.sessions
         ELSE NULL
    END AS conversion_rate

FROM base_sessions bs
LEFT JOIN pageviews pv
    ON bs.date = pv.date
   AND ( (bs.campaign_id = pv.campaign_id) OR (bs.campaign_id IS NULL AND pv.campaign_id IS NULL) )
   AND bs.utm_source = pv.utm_source
   AND bs.utm_medium = pv.utm_medium
   AND bs.utm_campaign = pv.utm_campaign
   AND bs.device_type = pv.device_type

LEFT JOIN conversions cv
    ON bs.date = cv.date
   AND ( (bs.campaign_id = cv.campaign_id) OR (bs.campaign_id IS NULL AND cv.campaign_id IS NULL) )
   AND bs.utm_source = cv.utm_source
   AND bs.utm_medium = cv.utm_medium
   AND bs.utm_campaign = cv.utm_campaign
   AND bs.device_type = cv.device_type

LEFT JOIN campaign_info ci
    ON bs.campaign_id = ci.campaign_id;